In [ ]:
'''
This notebook aggregates the natural disaster record from EM-DAT at the country level.
The main output is the number of hazard (per hazard type) for year 2000-2020, and the historical period (1900-1999).
The target disasters, that are likely to affect housing bulidings, are earthquake, flood, storm, fire, and volcanic eruption.
Death toll count and total damage are also included.
Data source: https://www.emdat.be/

'''


import pandas as pd
from pathlib import Path
import numpy as np
import warnings
import os
from google.colab import drive

# --- Pandas Display Settings ---
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 1000)

# ====================================================================
# 1. PATH AND GLOBAL CONFIGURATION 🌎
# ====================================================================

print("="*80)
print("DISASTER HISTORY ANALYSIS")
print("="*80)

# Mount Google Drive
drive.mount('/content/drive')

# --- Project Directory Setup ---
PROJECT_ROOT = Path("/content/drive/MyDrive/Resilient_Housing_Global_Regression")
DATA_RAW = PROJECT_ROOT / "data_raw"
DATA_PROCESSED = PROJECT_ROOT / "data_processed"

# --- Input Data Paths (Adjust filename if necessary) ---
EMDAT_SUBDIR = DATA_RAW / "EM-DAT"
EMDAT_FILE = EMDAT_SUBDIR / "public_emdat_custom_request_2025-11-07_f6f2cabe-ff99-4bc3-ba01-ab018fae62ec.xlsx"

# --- Output Path ---
COUNTRY_OUTPUT_CSV_FILE = DATA_PROCESSED / "country_disaster_summary_FINAL_PERIOD_SPLIT.csv"

# Year definition
YEAR_START_CURRENT = 2000
YEAR_END_CURRENT = 2020
YEAR_START_HISTORY = 1900
YEAR_END_HISTORY = 1999

# Ensure output directory exists
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

if not EMDAT_FILE.exists():
    print(f"❌ ERROR: File not found at {EMDAT_FILE}")
else:
    print(f" Target File: {EMDAT_FILE.name}")
    print(f" Main Period: {YEAR_START_CURRENT}-{YEAR_END_CURRENT}")
    print(f" History Period: {YEAR_START_HISTORY}-{YEAR_END_HISTORY}")

# --------------------------------------------------------------------
# Target disaster mapping
# --------------------------------------------------------------------
DISASTER_NAME_MAP = {
    'earthquake': 'EQK',
    'flood': 'FLD',
    'storm': 'STM',
    'volcanic activity': 'VOL',
    'wildfire': 'FIRE'
}

def shorten_disaster_col(original_col_name):
    lower_name = str(original_col_name).lower().strip()
    for full_name, short_code in DISASTER_NAME_MAP.items():
        if full_name == lower_name:
            return f"DIS_{short_code}"
    return "DROP_COL"

EXPECTED_SHORT_COLS = sorted([f"DIS_{code}" for code in DISASTER_NAME_MAP.values()])

DISASTER HISTORY ANALYSIS
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
 Target File: public_emdat_custom_request_2025-11-07_f6f2cabe-ff99-4bc3-ba01-ab018fae62ec.xlsx
 Main Period: 2000-2020
 History Period: 1900-1999


In [ ]:
# ====================================================================
# 2. LOAD AND PARSE EM-DAT DATA
# ====================================================================

print("\n[STEP 1] Loading and Parsing EM-DAT Data...")

emdat_df = pd.read_excel(EMDAT_FILE, header=0)

COL_MAP_EXCEL_TO_INTERNAL = {
    'DisNo.': 'dis_no',
    'ISO': 'iso_code',
    'Country': 'country_name',
    'Disaster Type': 'disaster_type',
    'Start Year': 'year',
    'Declaration': 'declaration',
    'Total Deaths': 'total_deaths',
    'Total Affected': 'total_affected',
    "Total Damage ('000 US$)": 'total_damage_usd'
}

# Column selection and renaming
emdat_raw = emdat_df[list(COL_MAP_EXCEL_TO_INTERNAL.keys())].rename(columns=COL_MAP_EXCEL_TO_INTERNAL)
emdat_raw['dis_no'] = emdat_raw['dis_no'].astype(str)

# Numeric conversions
for col in ['total_deaths', 'total_affected', 'total_damage_usd']:
    emdat_raw[col] = pd.to_numeric(emdat_raw[col], errors='coerce')

emdat_raw['year'] = pd.to_numeric(emdat_raw['year'], errors='coerce').astype('Int64')
emdat_raw['total_damage_usd'] = emdat_raw['total_damage_usd'] * 1000
emdat_raw['disaster_type'] = emdat_raw['disaster_type'].astype(str).str.strip()

print(f"✅ Total records loaded: {len(emdat_raw):,}")

# Cleansing
emdat_raw['iso_code'] = emdat_raw['iso_code'].astype(str).str.strip()
emdat_raw['country_name'] = emdat_raw['country_name'].astype(str).str.strip()
disasters_df = emdat_raw.copy()
disasters_df.dropna(subset=['iso_code', 'year'], inplace=True)
disasters_df = disasters_df[disasters_df['iso_code'] != '']

print(f"✅ Usable records (ISO & Year present): {len(disasters_df):,}")


[STEP 1] Loading and Parsing EM-DAT Data...
✅ Total records loaded: 12,986
✅ Usable records (ISO & Year present): 12,986


In [ ]:
# ====================================================================
# 3. AGGREGATE BY COUNTRY AND PERIOD
# ====================================================================

print("\n[STEP 2] Aggregating Data by Period...")

df_current = disasters_df[(disasters_df['year'] >= YEAR_START_CURRENT) & (disasters_df['year'] <= YEAR_END_CURRENT)].copy()
df_history = disasters_df[(disasters_df['year'] >= YEAR_START_HISTORY) & (disasters_df['year'] <= YEAR_END_HISTORY)].copy()

GROUPING_KEY = ['iso_code', 'country_name']

# --- A. Current Period (2000-2020) ---
total_disasters_country = df_current.groupby(GROUPING_KEY)['dis_no'].nunique().rename('DIS_TOTAL')
declared_disasters_country = df_current[df_current['declaration'].astype(str).str.upper() == 'YES'] \
    .groupby(GROUPING_KEY)['dis_no'].nunique().rename('DIS_DECLAR')

# Events per type
pivot_count = pd.pivot_table(df_current, index=GROUPING_KEY, columns='disaster_type', values='dis_no', aggfunc='nunique', fill_value=0)
pivot_count.rename(columns={col: shorten_disaster_col(col) for col in pivot_count.columns}, inplace=True)
pivot_count = pivot_count.reindex(columns=sorted(list(set(pivot_count.columns).union(EXPECTED_SHORT_COLS))), fill_value=0)
if 'DROP_COL' in pivot_count.columns: pivot_count.drop(columns='DROP_COL', inplace=True)

# Deaths per type
pivot_deaths_cur = pd.pivot_table(df_current, index=GROUPING_KEY, columns='disaster_type', values='total_deaths', aggfunc='sum', fill_value=0)
new_deaths_cols_cur = {col: f"{shorten_disaster_col(col)}_DEATHS" for col in pivot_deaths_cur.columns if shorten_disaster_col(col) != "DROP_COL"}
pivot_deaths_cur.rename(columns=new_deaths_cols_cur, inplace=True)
pivot_deaths_cur = pivot_deaths_cur[[c for c in pivot_deaths_cur.columns if "DIS_" in c]]

# Total Impact Current
impact_cur = df_current.groupby(GROUPING_KEY).agg(DIS_TOTAL_DEATHS=('total_deaths', 'sum'), DIS_TOTAL_DAMAGE_USD=('total_damage_usd', 'sum'))

# --- B. Historical Period (1900-1999) ---
impact_his = df_history.groupby(GROUPING_KEY).agg(DIS_TOTAL_DEATHS_History=('total_deaths', 'sum'), DIS_TOTAL_DAMAGE_USD_History=('total_damage_usd', 'sum'))

pivot_deaths_his = pd.pivot_table(df_history, index=GROUPING_KEY, columns='disaster_type', values='total_deaths', aggfunc='sum', fill_value=0)
new_deaths_cols_his = {col: f"{shorten_disaster_col(col)}_DEATHS_History" for col in pivot_deaths_his.columns if shorten_disaster_col(col) != "DROP_COL"}
pivot_deaths_his.rename(columns=new_deaths_cols_his, inplace=True)
pivot_deaths_his = pivot_deaths_his[[c for c in pivot_deaths_his.columns if "DIS_" in c]]


[STEP 2] Aggregating Data by Period...


In [ ]:
# ====================================================================
# 4. FINAL MERGE AND CALCULATION
# ====================================================================

print("\n[STEP 3] Final Merging and Calculation...")

country_summary = pd.concat([
    total_disasters_country,
    declared_disasters_country.reindex(total_disasters_country.index, fill_value=0),
    pivot_count,
    pivot_deaths_cur,
    impact_cur,
    impact_his,
    pivot_deaths_his
], axis=1).fillna(0)

# Avg Calculation
counts = country_summary['DIS_TOTAL']
country_summary['DIS_AVG_DEATHS'] = (country_summary['DIS_TOTAL_DEATHS'] / counts).replace([np.inf, -np.inf], 0).fillna(0).round(2)
country_summary['DIS_AVG_DAMAGE_USD'] = (country_summary['DIS_TOTAL_DAMAGE_USD'] / counts).replace([np.inf, -np.inf], 0).fillna(0).round(2)

# Type casting to Int for specific columns
int_cols = ['DIS_TOTAL', 'DIS_DECLAR'] + list(pivot_count.columns) + list(pivot_deaths_cur.columns) + \
           ['DIS_TOTAL_DEATHS', 'DIS_TOTAL_DAMAGE_USD', 'DIS_TOTAL_DEATHS_History', 'DIS_TOTAL_DAMAGE_USD_History'] + \
           list(pivot_deaths_his.columns)

for col in int_cols:
    if col in country_summary.columns:
        country_summary[col] = country_summary[col].astype(int)

country_summary.reset_index(inplace=True)

# Save
country_summary.to_csv(COUNTRY_OUTPUT_CSV_FILE, index=False, encoding='utf-8-sig')

print("="*80)
print(f"📊 FINAL STATS:")
print(f" - Countries processed: {len(country_summary)}")
print(f" - File saved to: {COUNTRY_OUTPUT_CSV_FILE}")
print("="*80)
display(country_summary.head())


[STEP 3] Final Merging and Calculation...
📊 FINAL STATS:
 - Countries processed: 223
 - File saved to: /content/drive/MyDrive/Resilient_Housing_Global_Regression/data_processed/country_disaster_summary_FINAL_PERIOD_SPLIT.csv


,iso_code,country_name,DIS_TOTAL,DIS_DECLAR,DIS_EQK,DIS_FIRE,DIS_FLD,DIS_STM,DIS_VOL,DIS_EQK_DEATHS,DIS_FLD_DEATHS,DIS_STM_DEATHS,DIS_VOL_DEATHS,DIS_FIRE_DEATHS,DIS_TOTAL_DEATHS,DIS_TOTAL_DAMAGE_USD,DIS_TOTAL_DEATHS_History,DIS_TOTAL_DAMAGE_USD_History,DIS_EQK_DEATHS_History,DIS_FLD_DEATHS_History,DIS_STM_DEATHS_History,DIS_VOL_DEATHS_History,DIS_FIRE_DEATHS_History,DIS_AVG_DEATHS,DIS_AVG_DAMAGE_USD
0,AFG,Afghanistan,99,0,15,0,76,8,0,1455,2826,495,0,0,4776,28050000,12579,430010000,10100,2469,10,0,0,48.24,2.833333e+05
1,AGO,Angola,40,2,0,0,40,0,0,0,786,0,0,0,786,12000000,0,0,0,0,0,0,0,19.65,3.000000e+05
2,AIA,Anguilla,1,0,0,0,0,1,0,0,0,4,0,0,4,200000000,5,35050000,0,0,5,0,0,4.00,2.000000e+08
3,ALB,Albania,18,4,4,1,11,2,0,51,8,8,0,0,67,771573000,614,55800000,575,39,0,0,0,3.72,4.286517e+07
4,ARE,United Arab Emirates,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.00,0.000000e+00
